# Part 2- 11,000+ images

# Install Required Libraries
In this cell, we install all libraries required for building the retrieval pipeline.
It includes FAISS for vector indexing, HuggingFace Transformers for FashionCLIP and BLIP models, and supporting utilities.

In [ ]:
!pip install -q transformers faiss-cpu torch torchvision pillow tqdm matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 116.2 MB/s eta 0:00:00


# Import Packages
This cell imports all the Python libraries used in the project such as FAISS, PyTorch, Transformers, PIL, and numpy.

In [ ]:
import os, pickle, random, zipfile, re
import numpy as np
from tqdm import tqdm
from PIL import Image
import cv2

import torch
import faiss
from transformers import CLIPProcessor, CLIPModel
from transformers import BlipProcessor, BlipForConditionalGeneration, BlipForImageTextRetrieval



# Dataset Setup and Extraction

In [ ]:
zip_path = "/content/Glance_Train2.zip"
extract_path = "/content/dataset/train"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted to:", extract_path)


✅ Extracted to: /content/dataset/train


# Collect All Image Paths from Dataset Directory

In [ ]:
def get_all_images(folder):
    valid = (".jpg", ".jpeg", ".png", ".webp")
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(valid):
                paths.append(os.path.join(root, f))
    return paths


# Load FashionCLIP for Embedding Extraction
This cell loads FashionCLIP, a fashion-domain aligned CLIP model.
It converts both images and text queries into embeddings in the same vector space, enabling semantic retrieval.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

FASHIONCLIP_NAME = "patrickjohncyh/fashion-clip"  # FashionCLIP
clip_model = CLIPModel.from_pretrained(FASHIONCLIP_NAME).to(device)
clip_processor = CLIPProcessor.from_pretrained(FASHIONCLIP_NAME)

print("✅ FashionCLIP loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

✅ FashionCLIP loaded


# Load BLIP Captioning and Cross-Encoder Models
BLIP Captioning is used to generate image descriptions which help extract attributes like clothing type, color and context.

In [ ]:
BLIP_CAPTION_NAME = "Salesforce/blip-image-captioning-base"
blip_cap_model = BlipForConditionalGeneration.from_pretrained(BLIP_CAPTION_NAME).to(device)
blip_cap_processor = BlipProcessor.from_pretrained(BLIP_CAPTION_NAME)

print("✅ BLIP caption model loaded")


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

✅ BLIP caption model loaded


BLIP ITM (Image-Text Matching) is used as a cross-encoder reranker to boost final retrieval quality.

In [ ]:
BLIP_ITM_NAME = "Salesforce/blip-itm-base-coco"
itm_model = BlipForImageTextRetrieval.from_pretrained(BLIP_ITM_NAME).to(device)
itm_processor = BlipProcessor.from_pretrained(BLIP_ITM_NAME)

print("✅ BLIP ITM cross-encoder loaded")


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/895M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/895M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

✅ BLIP ITM cross-encoder loaded


# Region-Based Color Extraction (Top/Bottom Matching)

In [ ]:
def dominant_color_name(rgb):
    # Very simple mapping (enough for internship)
    r, g, b = rgb
    if r > 160 and g < 120 and b < 120: return "red"
    if b > 160 and r < 120 and g < 160: return "blue"
    if g > 160 and r < 140 and b < 140: return "green"
    if r > 180 and g > 180 and b > 180: return "white"
    if r < 60 and g < 60 and b < 60: return "black"
    if r > 160 and g > 140 and b < 120: return "yellow"
    return "unknown"

def region_colors(pil_img: Image.Image):
    img = np.array(pil_img)
    h = img.shape[0]

    top = img[:h//2, :, :]
    bottom = img[h//2:, :, :]

    def mean_rgb(region):
        mean = region.reshape(-1, 3).mean(axis=0)
        return tuple(mean.astype(int))

    top_color = dominant_color_name(mean_rgb(top))
    bot_color = dominant_color_name(mean_rgb(bottom))
    return {"top_color": top_color, "bottom_color": bot_color}


# Image Feature Extraction (FashionCLIP) + Caption Generation (BLIP)
This cell defines helper functions to extract normalized FashionCLIP image embeddings and generate descriptive captions using BLIP.
The embeddings are used for FAISS indexing, while captions provide fine-grained semantic attributes (color, clothing type, context) for reranking

In [ ]:
def clip_image_embedding(img: Image.Image):
    inputs = clip_processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        feat = clip_model.get_image_features(**inputs)
    feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy().astype("float32")[0]

def blip_caption(img: Image.Image):
    inputs = blip_cap_processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        out = blip_cap_model.generate(**inputs, max_new_tokens=30)
    return blip_cap_processor.decode(out[0], skip_special_tokens=True)


# Define Fashion Vocabulary & Attribute Extraction
In this cell, we define the vocabulary of key fashion entities such as colors, clothing types (topwear, bottomwear, one-piece) and accessories.

In [ ]:
COLORS = ["red","blue","green","black","white","yellow","pink","orange","purple","brown","grey","gray","beige"]

TOP_CLOTHES = sorted(set([
    "shirt", "tshirt", "t-shirt", "tee", "polo", "blouse", "top", "tank top", "crop top",
    "camisole", "sweatshirt", "hoodie", "sweater", "cardigan",
    "jacket", "blazer", "coat", "raincoat", "overcoat",
    "kurti", "kurta", "tunic"
]))

BOTTOM_CLOTHES = sorted(set([
    "pants", "jeans", "denim", "trousers", "chinos", "shorts", "skirt",
    "leggings", "joggers", "track pants",
    "salwar", "churidar", "palazzo"
]))

ONE_PIECE = sorted(set([
    "dress", "gown", "frock", "maxi", "midi dress", "minidress", "shift dress", "bodycon",
    "jumpsuit", "romper", "playsuit",
    "saree", "sari", "lehenga", "anarkali", "kameez", "salwar suit",
    "onesie", "overall", "overalls", "dungaree", "dungarees"
]))

ACCESSORIES = sorted(set([
    "tie", "belt", "scarf", "cap", "hat", "sunglasses", "watch", "shoes", "sneakers"
]))


SCENES = ["street","beach","office","wedding","park","mall","indoor","outdoor"]

ALL_CLOTHES = sorted(set(TOP_CLOTHES + BOTTOM_CLOTHES + ONE_PIECE + ACCESSORIES))


def extract_attributes(text):
    text = text.lower()

    attrs = {"colors": [], "clothes": [], "scene": []}

    # colors
    for c in COLORS:
        if c in text:
            attrs["colors"].append(c)

    # clothes (now covers ALL categories)
    for t in ALL_CLOTHES:
        if t in text:
            attrs["clothes"].append(t)

    # scenes
    for s in SCENES:
        if s in text:
            attrs["scene"].append(s)

    return attrs


# Indexer: Build Embeddings + Captions + Metadata for Dataset Part

This cell processes all images in the selected training directory and extracts FashionCLIP embeddings for vector search.

It also generates BLIP captions and derives structured attributes (color, clothing type, scene) along with region-based color metadata.

The outputs form the complete searchable representation for this dataset chunk and will later be stored in FAISS

In [ ]:
TRAIN_DIR = "/content/dataset/train/Glance_Train2"
INDEX_DIR = "/content/index2"
os.makedirs(INDEX_DIR, exist_ok=True)

image_paths = get_all_images(TRAIN_DIR)
random.shuffle(image_paths)
# image_paths = image_paths[:900]  # ✅ test run first

embeddings = []
captions = {}
attributes = {}
region_meta = {}

for p in tqdm(image_paths):
    try:
        img = Image.open(p).convert("RGB")
    except:
        continue

    # ❌ No YOLO cropping — use image directly
    img_crop = img

    # ✅ FashionCLIP embedding
    emb = clip_image_embedding(img_crop)
    embeddings.append(emb)

    # ✅ BLIP caption
    cap = blip_caption(img_crop)
    captions[p] = cap

    # ✅ Attrs + region colors
    attrs = extract_attributes(cap)
    attributes[p] = attrs
    region_meta[p] = region_colors(img_crop)

embeddings = np.vstack(embeddings).astype("float32")
print("✅ Embeddings:", embeddings.shape)


100%|██████████| 11182/11182 [52:54<00:00,  3.52it/s]

✅ Embeddings: (11182, 512)


# Save FAISS Index and Metadata Files (Part-wise Storage)

In [ ]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

faiss.write_index(index, os.path.join(INDEX_DIR, "faiss_index2.bin"))

with open(os.path.join(INDEX_DIR, "image_paths2.pkl"), "wb") as f:
    pickle.dump(image_paths, f)

with open(os.path.join(INDEX_DIR, "captions2.pkl"), "wb") as f:
    pickle.dump(captions, f)

with open(os.path.join(INDEX_DIR, "attributes2.pkl"), "wb") as f:
    pickle.dump(attributes, f)

with open(os.path.join(INDEX_DIR, "region_meta2.pkl"), "wb") as f:
    pickle.dump(region_meta, f)

print("✅ Saved complete index to:", INDEX_DIR)


✅ Saved complete index to: /content/index2


# Export Index Folder as ZIP for Download

In [ ]:
import shutil
from google.colab import files

# Path of folder you want to download
folder_to_zip = "/content/index2"     # change if your folder is elsewhere
zip_name = "index_folder2"

# Create zip
shutil.make_archive(zip_name, 'zip', folder_to_zip)

# Download zip
files.download(zip_name + ".zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>